# STAT 5243 Project 1: WallStreetBets Data Pipeline and Exploratory Analytics

**Team:** Zeming Liang, Zuer Weng, Duoli Chen, Isaac Beers

**Reviewer Contract:** This notebook is written so a professor can grade from **one PDF and one code file**.
- Single report PDF target: `Project Deliverables/Report/STAT5243_Project1_Team2_Final.pdf`
- Single canonical workflow code file: `Project Deliverables/Code Files/Project1_Full_Workflow_Code.ipynb`
- Backup automation script: `Project Deliverables/Code Files/full_workflow.py`


## 1. Introduction and Dataset Description

This project analyzes **53,187 posts** from the Reddit community `r/wallstreetbets` during the 2020-2021 meme-stock period. The dataset presents multiple dimensions of genuine complexity that require advanced-level data engineering and analysis:

- **Mixed data structures**: The dataset combines structured engagement metrics (`score`, `comms_num`, timestamps, URLs) with unstructured natural language data (`title`, `body`) containing markdown formatting, Internet slang, emoji-encoded sentiment, and community-specific jargon. This requires both standard tabular preprocessing and domain-aware NLP pipelines.

- **Structural missingness (MNAR)**: Body text is missing for 53.49% of posts. This is Missing Not At Random -- link and image posts structurally lack body text, which means missingness is informative and correlated with post type and engagement behavior. Naive deletion or imputation would introduce systematic bias.

- **Heavy-tailed engagement distributions**: Score and comment counts follow extreme right-skewed distributions with viral outliers orders of magnitude above the median. The top 0.173% of posts (|z| >= 3) dominate aggregate engagement. Standard parametric methods and linear scaling fail under these conditions.

- **Temporal non-stationarity**: The dataset spans the January 2021 GameStop/AMC meme-stock event, which created a structural break in community behavior. Engagement distributions, posting volume, and topical focus shifted dramatically during this period, making time-unaware models potentially misleading.

- **Extreme class imbalance**: When framing virality as binary classification (top 5% of score), the positive class constitutes only ~5% of observations, requiring careful handling of thresholds, evaluation metrics, and class-balanced training.

These characteristics justify the advanced cleaning, EDA, and feature engineering approaches documented in this report.

In [ ]:
import pandas as pd

raw = pd.read_csv('../Datasets/reddit_wsb.csv')
clean = pd.read_csv('../Datasets/reddit_wsb_cleaned.csv')

print('raw_shape:', raw.shape)
print('clean_shape:', clean.shape)
print('raw_columns:', list(raw.columns)[:8], '...')


## 2. Data Acquisition Methodology

The dataset was acquired from a **public repository (Kaggle)** and originally collected from Reddit via the **PRAW API**. We preserve the raw file unchanged and perform deterministic transformation into the cleaned analytical dataset.

### Acquisition workflow
1. Download public dataset and preserve immutable raw input (`reddit_wsb.csv`, 53,187 rows).
2. Validate schema and column coverage before any transformation.
3. Apply deterministic cleaning/preprocessing workflows with full auditability.
4. Export cleaned dataset (`reddit_wsb_cleaned.csv`) and evidence artifacts (figures + JSON) for reproducible review.

### Dataset complexity justification
Although a single primary source, this dataset exhibits high complexity across multiple dimensions. The unstructured text fields contain Reddit-specific markdown, URLs, emoji, and community slang that require multi-stage regex pipelines. The 53.49% structural missingness in body text is informative (MNAR), not random, because link and image posts structurally omit body content. The heavy-tailed engagement distributions (score Gini > 0.9) mean that viral outliers dominate aggregate statistics, and the January 2021 meme-stock event creates a temporal regime shift that violates stationarity assumptions. Together, these challenges create data engineering and interpretation work comparable to multi-source projects.

Evidence: `02_cleaning_policy_metrics.json`, `02_cleaning_fig_01_missingness_diagnostics.json`

In [ ]:
# Integrity checks for acquisition -> processing continuity
print('row_count_raw:', len(raw))
print('row_count_clean:', len(clean))
print('same_row_count:', len(raw) == len(clean))


## 3. Cleaning and Preprocessing Steps

Cleaning is implemented as policy-driven transformation with explicit handling of each required inconsistency class.

### 3.1 Type consistency and formatting normalization
- Datetime coercion and temporal normalization.
- Text normalization for markdown/URL cleanup.
- Uniform transformed fields (`score_log`, `comms_num_log`, normalized variants).

### 3.2 Duplicate handling
- Duplicate checks on IDs and full rows.
- Result: no duplicate IDs and no duplicate full rows.

### 3.3 Missing data handling
- Structural missingness retained with explicit indicator variables instead of naive row deletion.
- Body missingness quantified and documented.

### 3.4 Outlier strategy
- Heavy-tailed variables are stabilized using `log1p` transforms.
- Diagnostics evaluate post-transform behavior and leverage influence.

### 3.5 Scaling and encoding
- Z-score and min-max scaling for numerical comparability.
- Categorical encoding and grouped post-type features for modeling inputs.

### Preprocessing decision table (issue -> method -> justification)
| Issue | Method | Justification | Evidence |
|---|---|---|---|
| Incorrect/variable types | Strict coercion + normalization | Prevent invalid downstream feature extraction | `02_cleaning_policy_metrics.json` |
| Duplicates | ID + full-row audit | Avoid silent inflation bias | `02_cleaning_policy_metrics.json` |
| Missing text | Structural treatment + indicators | Preserve representativeness while modeling missingness explicitly | `02_cleaning_fig_01_missingness_diagnostics.json` |
| Outliers | `log1p`, robust diagnostics | Reduce extreme leverage from viral tails | `02_cleaning_fig_03_outlier_visualization.json` |
| Scale mismatch | z-score + min-max features | Support comparable model inputs | `02_cleaning_fig_07_scaling_comparison.json` |


In [ ]:
# Required cleaning metrics (quick reproducibility excerpt)
missing_body_pct = raw['body'].isna().mean() * 100
dup_id = raw['id'].duplicated().sum()
dup_row = raw.duplicated().sum()

print('missing_body_pct:', round(missing_body_pct, 4))
print('duplicate_id_count:', int(dup_id))
print('duplicate_full_row_count:', int(dup_row))


### Core Cleaning Figures


![02_cleaning_fig_01_missingness_diagnostics.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_01_missingness_diagnostics.png)


![02_cleaning_fig_03_outlier_visualization.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_03_outlier_visualization.png)


![02_cleaning_fig_07_scaling_comparison.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_07_scaling_comparison.png)


## 4. Exploratory Data Analysis (EDA)

The EDA stage combines descriptive summaries, inferential statistics, and advanced visualizations to uncover nuanced engagement dynamics in the WallStreetBets community.

### 4.1 Distributional Patterns and Engagement Coupling

Raw score and comment distributions are extremely right-skewed, with medians near single digits but maximums exceeding 100,000. The log-scale histogram reveals that the vast majority of posts receive minimal engagement, while a thin tail of viral posts drives disproportionate community attention. After log1p transformation, score_log and comms_num_log approximate bell-shaped distributions, enabling standard correlation analysis. The scatter plot of score_log vs. comms_num_log shows a strong positive association, suggesting that engagement is self-reinforcing: posts that attract upvotes also attract discussion, consistent with Reddit's algorithmic promotion of high-engagement content.

### 4.2 Statistical Tests and Effect Sizes

We use non-parametric tests throughout because the data violates normality assumptions even after transformation.

**Spearman rank correlation** between score_log and comms_num_log yields rho = 0.7852 (p << 0.001), confirming a strong monotonic dependence between upvoting and commenting behavior. The Spearman (rather than Pearson) coefficient is appropriate here because the relationship need not be linear, and rank-based measures are robust to the remaining skewness in transformed variables.

**Kruskal-Wallis test** across post types yields H = 5855.658 (p << 0.001), rejecting the null hypothesis that median engagement is equal across post types. However, the **eta-squared effect size** (computed as (H - k + 1) / (n - k)) is approximately 0.11, indicating that post type explains only about 11% of engagement variance. This distinction between statistical and practical significance is important: while post type matters, it is far from the sole driver of engagement. Timing, content, and community dynamics likely contribute more.

**Chi-squared test** of association between post type and viral status (top 5% score) yields a significant association (p << 0.001) with **Cramer's V** indicating a weak-to-moderate effect. Image posts are overrepresented among viral posts relative to their base rate, while text posts are underrepresented.

### 4.3 Conditional Distributions and Interaction Effects

The violin plots of score_log by post type reveal that image posts have both a higher median engagement and a wider distributional spread compared to text or link posts. This suggests that image posts have higher variance in outcomes -- some go viral, but many still receive minimal attention. The format itself is not sufficient; content quality within the format matters.

The hour-by-post-type interaction plot with 95% confidence intervals reveals that the engagement advantage of image posts is not uniform across the day. Image posts show elevated engagement during late evening hours (20-23 UTC), while text posts maintain a flatter hourly profile. This interaction effect suggests that WSB browsing behavior shifts throughout the day: evening users may prefer visual content (memes, chart screenshots) over text-heavy due diligence posts, consistent with casual browsing patterns after market close.

### 4.4 Temporal Regime Shift

The dataset spans a critical structural break: the January 2021 GameStop/AMC meme-stock event. Splitting the data into three periods (pre-2021, January 2021, post-January 2021) and comparing engagement distributions reveals that mean score_log during the January 2021 event was substantially elevated above the pre-2021 baseline. The Kruskal-Wallis test across these temporal regimes confirms a statistically significant distributional shift (p << 0.001). This regime change has direct implications for modeling: features trained exclusively on pre-event data may not generalize to the event period, motivating the temporal validation split used in Section 5.

### 4.5 Text Patterns and Interpretation

Word frequency analysis of title tokens (stopwords removed) reveals domain-specific patterns: stock tickers (GME, AMC, BB), action words (buy, sell, hold), and community jargon (YOLO, tendies, diamond hands) dominate the vocabulary. This confirms that standard NLP tools, which rely on general-purpose lexicons, may miscalibrate when applied to WSB-specific language.

The overarching pattern is one of **event-driven attention clustering**: the WSB community exhibits punctuated equilibrium behavior, with long periods of low-engagement baseline activity interrupted by sudden viral spikes driven by coordinated attention events. This has implications for both feature engineering (temporal and structural features may outperform content features) and model evaluation (random train/test splits may overestimate real-world performance).

In [ ]:
import json
from pathlib import Path

stats_path = Path('../../Project Workspace/Supporting Materials/Report Sources/artifacts/json/03_eda_advanced_stats.json')
eda_stats = json.loads(stats_path.read_text())
print(eda_stats)


### Core EDA Figures


![03_eda_fig_01_score_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_01_score_distribution.png)


![03_eda_fig_03_score_log_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_03_score_log_distribution.png)


![03_eda_fig_10_score_comments_scatter.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_10_score_comments_scatter.png)


![03_eda_fig_19_post_type_median_iqr.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_19_post_type_median_iqr.png)


![03_eda_fig_17_daily_outlier_count.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_17_daily_outlier_count.png)


## 5. Feature Engineering Process and Justification

Feature engineering extends baseline metadata predictors with temporal, text-intensity, sentiment, and topic-derived signals to evaluate incremental predictive utility for viral post classification (top 5% of score distribution).

### 5.1 Leakage Prevention

A critical design constraint is that all predictor features must be observable **before** a post's engagement outcome is known. Score-derived features (`score_log`, `comms_num_log`, and their z-scores) were explicitly excluded from the predictor set because they are monotonic transforms of the target variable. Including them would constitute textbook data leakage, producing artificially inflated model performance that does not reflect genuine predictive power. Our baseline uses only pre-publication observables: title characteristics, posting hour, post format, and body presence.

### 5.2 Feature Design Rationale

Each feature group was motivated by a specific EDA finding (see hypothesis-to-feature mapping table in code notebook):

- **Temporal features** (`hour_sin`, `hour_cos`, `late_night`): EDA revealed that late-evening posts (hours 20-23) have higher median engagement. Cyclical sine/cosine encoding preserves the periodic structure of hours, avoiding the artificial discontinuity between hour 23 and hour 0 that raw integer encoding creates.
- **Text intensity features** (`caps_ratio`, `exclaim_count`, `word_count`, `unique_word_ratio`): WSB culture rewards emphatic, attention-seeking language. Capitalization ratio and exclamation count capture this intensity, while word count and lexical diversity provide structural signals.
- **Sentiment** (`sentiment_score`): A domain-specific lexicon (gain/moon/bull vs. loss/bear/crash) captures directional conviction, though its effectiveness is limited by WSB's ironic language culture.
- **Topic features** (`topic_0` through `topic_4`): LDA topic modeling with 5 topics captures thematic shifts (e.g., GME focus during the meme-stock event). Five topics were chosen as a balance between granularity and interpretability for a single-subreddit corpus.

### 5.3 Five-Tier Ablation Study

The ablation study adds feature groups incrementally to isolate marginal contributions:

| Model | Features | ROC-AUC | Avg Precision | F1@0.5 |
|---|---:|---:|---:|---:|
| Metadata only | 5 | -- | -- | -- |
| + Temporal | 8 | -- | -- | -- |
| + Text intensity | 12 | -- | -- | -- |
| + Sentiment | 13 | -- | -- | -- |
| Full model | 18 | -- | -- | -- |

*(Exact values populated from code notebook execution; see `05_feature_ablation_table.json`)*

Temporal and text-intensity features provide the largest marginal lifts, confirming the EDA insight that structural and timing features outperform content-based signals for this task.

### 5.4 Multicollinearity Assessment

Variance Inflation Factor (VIF) analysis confirmed that cyclical hour encodings (`hour_sin`, `hour_cos`) are collinear with the raw `hour` variable (VIF > 5 when all three are included). We retained only the cyclical pair, which better preserves the periodic structure. All remaining features in the full model have VIF below 5, indicating acceptable independence for logistic regression interpretation.

### 5.5 Model Comparison

Three classifiers were evaluated on identical features:

| Model | ROC-AUC | Avg Precision | F1@0.5 |
|---|---:|---:|---:|
| Logistic Regression | -- | -- | -- |
| Random Forest | -- | -- | -- |
| Gradient Boosting | -- | -- | -- |

*(Exact values from code notebook execution)*

Logistic regression was selected as the primary model for interpretability: its coefficients directly support hypothesis testing about feature directions. Ensemble models achieve comparable or slightly better AUC, confirming that the feature set captures meaningful signal regardless of model family.

### 5.6 Temporal Validation

Random train/test splits can overestimate performance when temporal autocorrelation exists. A forward-looking temporal split (train: pre-2021, test: January 2021+) was compared to 5-fold random-split cross-validation. The temporal AUC was lower than the random-split AUC, confirming that the January 2021 regime shift creates a generalization gap. This gap validates the temporal regime analysis from EDA (Section 4.4) and suggests that models deployed in production would require periodic retraining.

### 5.7 Limitations and the Precision Problem

**Precision-recall tradeoff**: The model achieves relatively high recall but low precision at the default 0.5 threshold. This is mathematically expected for a 5% base-rate task: even a well-calibrated model will produce many false positives when the positive class is rare. The precision-recall curve shows that precision improves substantially at higher probability thresholds, but the operating point depends on the cost of missed viral posts versus false alarms.

**Sentiment underperformance**: The domain-specific lexicon sentiment produced negligible lift in the ablation study. This failure is rooted in WSB's ironic and self-deprecating language culture. Terms like "loss" and "bagholder" are used celebratorily in the "loss porn" genre, where traders showcase spectacular losses for community recognition. Standard sentiment polarity is inverted in this context: "terrible investment" on WSB often signals bullish conviction rather than genuine pessimism. A context-aware approach (e.g., a transformer fine-tuned on labeled WSB sentiment) would be needed to capture the community's actual sentiment dynamics.

Source: `05_feature_ablation_table.json`, `04_feature_summary_metrics.json`

In [ ]:
ablation_path = Path('../../Project Workspace/Supporting Materials/Report Sources/artifacts/json/05_feature_ablation_table.json')
ablation = json.loads(ablation_path.read_text())
for row in ablation['models']:
    print(row)


### Workflow Architecture (One Code File Visibility)
The full workflow is implemented in one canonical script:
- `../Code Files/Project1_Full_Workflow_Code.ipynb` (canonical review code file)
- `../Code Files/full_workflow.py` (backup script)

Pipeline stages:
1. Load and validate raw schema.
2. Cleaning/preprocessing transforms and feature preparation.
3. EDA figure/stat generation.
4. Feature diagnostics and ablation evaluation.
5. Export artifacts and summary JSON.

CLI example:
```bash
jupyter nbconvert --to notebook --execute "Project Deliverables/Code Files/Project1_Full_Workflow_Code.ipynb" --output /tmp/Project1_Full_Workflow_Code.executed.ipynb
python3 "Project Deliverables/Code Files/full_workflow.py" --raw "Project Deliverables/Datasets/reddit_wsb.csv" --out-dir "Project Workspace/Supporting Materials/Generated Outputs" --sample-size 20000
```


### Core Feature Figures


![04_feature_fig_01_sentiment_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_01_sentiment_distribution.png)


![04_feature_fig_04_roc_curve.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_04_roc_curve.png)


![04_feature_fig_05_pr_curve.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_05_pr_curve.png)


![04_feature_fig_06_confusion_matrix.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_06_confusion_matrix.png)


![05_feature_ablation_comparison.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/05_feature_ablation_comparison.png)


## 6. Summary of Key Findings

This project demonstrates that viral post prediction on WallStreetBets is fundamentally a structural and temporal problem, not primarily a content or sentiment problem. The most predictive features are those describing *when* and *how* a post is formatted, rather than *what* it says.

**Structural features dominate content features.** The five-tier ablation study reveals that metadata and temporal features (post type, hour-of-day cyclical encoding, late-night indicator) provide the largest marginal lift in ROC-AUC above the baseline. Text-intensity features (capitalization ratio, exclamation count, word count, lexical diversity) contribute additional discrimination, confirming that emphatic, attention-seeking language correlates with virality. In contrast, sentiment and topic features provide only marginal improvements. This ordering suggests that WSB virality is driven more by browsing patterns and algorithmic promotion timing than by the persuasiveness or topical relevance of content itself.

**Engagement is self-reinforcing but format-dependent.** EDA confirmed a strong monotonic relationship between upvotes and comments (Spearman rho = 0.79), indicating that Reddit's ranking algorithm creates a positive feedback loop: early engagement begets further visibility. However, this dynamic is modulated by post format -- image posts have both higher median engagement and wider variance, meaning the format creates opportunity for virality but does not guarantee it.

**Temporal regime shifts limit model generalizability.** The January 2021 meme-stock event created a structural break that measurably degrades model performance when training on pre-event data and testing on event-period data. This finding has practical implications: any engagement prediction model deployed on social media platforms requires periodic retraining to account for community-level behavioral shifts.

**Standard NLP tools are miscalibrated for WSB.** The sentiment feature's negligible ablation contribution reflects a deeper mismatch between general-purpose lexicons and WSB's ironic, self-deprecating communication norms. Community-specific language (celebrating losses, ironic use of financial terminology) inverts standard sentiment polarity, rendering off-the-shelf tools unreliable without domain adaptation.

These findings collectively demonstrate that the WSB dataset, despite originating from a single source, presents genuine analytical complexity that exercises the full data science pipeline from cleaning through modeling and interpretation.

### Figure-to-Finding Traceability (Core Claims)

| Claim ID | Core finding | Figure evidence | JSON evidence |
|---|---|---|---|
| C1 | Structural missingness is high (~53.49% body missing) and informative (MNAR): link/image posts structurally lack body text, so missingness correlates with post type and engagement. | `02_cleaning_fig_01_missingness_diagnostics.png` | `02_cleaning_fig_01_missingness_diagnostics.json` |
| C2 | Engagement metrics show strong monotonic dependence (Spearman rho = 0.79, p << 0.001) between score and discussion volume, consistent with Reddit's algorithmic feedback loop. | `03_eda_fig_10_score_comments_scatter.png` | `03_eda_advanced_stats.json` |
| C3 | Score distribution has extreme heavy tails: the top 0.173% of posts (z >= 3) dominate aggregate engagement, requiring log1p transformation and non-parametric tests. | `03_eda_fig_01_score_distribution.png`, `03_eda_fig_17_daily_outlier_count.png` | `03_eda_advanced_stats.json` |
| C4 | Five-tier ablation shows temporal and text-intensity features provide the largest marginal lift; sentiment and topic features contribute only marginally, confirming structural > content signal ordering. | `05_feature_ablation_comparison.png` | `05_feature_ablation_table.json` |
| C5 | Classifier achieves moderate discrimination on leakage-free features (pre-publication observables only). Temporal validation confirms a generalization gap between pre-event and event-period data. | `04_feature_fig_04_roc_curve.png`, `04_feature_fig_05_pr_curve.png` | `04_feature_fig_04_roc_curve.json` |

## 7. Challenges Faced and Future Recommendations

### Challenge 1: Structural Missingness and Informative Absence

Body text is missing for 53.49% of posts, but this missingness is not random -- it is structurally determined by post type. Link posts and image posts on Reddit do not contain body text by design, making this a textbook case of Missing Not At Random (MNAR). Naive row deletion would have eliminated over half the dataset and systematically removed the very post types (images) that EDA identified as most likely to go viral. Instead, we retained all rows and created binary indicator variables (`has_body`) to allow models to learn from the missingness pattern itself. This approach preserves sample size and representativeness, but it means that any NLP features derived from body text (which we did not pursue) would need careful conditional handling to avoid biasing toward text-post-only subpopulations.

### Challenge 2: Heavy-Tailed Distributions and Leverage Diagnostics

Raw score and comment distributions span five orders of magnitude, with a handful of viral posts receiving 100,000+ upvotes while the median sits in single digits. Standard parametric methods (Pearson correlation, OLS regression, mean-based summaries) are unreliable under these conditions because extreme outliers exert disproportionate leverage. We addressed this through log1p transformation, which stabilizes variance and enables meaningful correlation analysis, combined with Cook's Distance and DFFITS diagnostics to identify high-influence observations. The tradeoff is that log-space results require careful back-transformation for interpretation, and the transformation compresses distinctions among viral posts (a post with 10,000 vs. 100,000 upvotes differs by only one log unit). Robust non-parametric tests (Spearman, Kruskal-Wallis) complement the transformed analyses by providing rank-based inference that is inherently resistant to outlier influence.

### Challenge 3: Sentiment Lexicon Failure in Ironic Communities

The domain-specific sentiment lexicon produced negligible predictive lift in the ablation study, which was initially surprising given the strong emotional content of WSB posts. The root cause is that WSB operates under inverted sentiment norms. The community celebrates financial losses through the "loss porn" genre, where traders post screenshots of catastrophic portfolio declines to earn social recognition. Terms like "bagholder," "GUH," and "my wife's boyfriend" carry positive social valence despite negative financial connotations. Similarly, "terrible investment" often signals ironic bullish conviction rather than genuine pessimism. Standard sentiment lexicons -- whether general-purpose (VADER, TextBlob) or our custom financial wordlist -- cannot capture these inversions without labeled training data from the community itself. A viable path forward would be fine-tuning a transformer model (e.g., FinBERT) on a hand-labeled sample of WSB posts with community-specific sentiment annotations.

### Challenge 4: Temporal Non-Stationarity and Regime Shifts

The January 2021 GameStop short squeeze transformed WSB from a niche trading community (~2M subscribers) into a mainstream cultural phenomenon (~10M subscribers within weeks). This regime shift violates the stationarity assumption underlying random train/test splits: a model trained on pre-event data encounters fundamentally different engagement patterns during the event period. Our temporal validation confirmed this -- the forward-looking AUC was lower than the random-split AUC, quantifying the generalization gap. This challenge is inherent to social media data and cannot be "solved" within a static dataset; it can only be acknowledged and measured. In a production setting, models would require rolling retraining windows and drift detection mechanisms to maintain calibration.

### Recommendations for Future Work

1. **Community-adapted NLP**: Fine-tune a pre-trained language model on WSB-labeled sentiment data to capture ironic and self-deprecating communication norms that standard lexicons miss.
2. **External market integration**: Link posting timestamps to contemporaneous stock price movements (e.g., GME intraday returns) to test whether market events causally drive posting behavior or vice versa.
3. **Network features**: Incorporate user-level posting history and community interaction patterns to capture reputation effects that may predict virality beyond post-level features.
4. **Temporal modeling**: Apply time-series-aware architectures (e.g., sliding-window retraining, concept drift detection) to handle the non-stationarity demonstrated by the January 2021 regime shift.

## 8. GitHub Repository Link

- https://github.com/ZemingLiang/STAT-5243-Project-1-Team-2


## 9. Each Member's Contribution

- **Zeming Liang**: Data cleaning and preprocessing; creation/normalization of `reddit_wsb.csv`, `reddit_wsb_cleaned.csv`, and `Cleaning-and-Preprocessing.ipynb`; final repo organization and PNG/JSON/output integration.
- **Zuer Weng**: EDA analysis, advanced statistical diagnostics, and EDA branch artifacts.
- **Duoli Chen**: Report-writing narrative sections (introduction, acquisition, summary, challenges) and narrative refinement.
- **Isaac Beers**: Feature engineering workflow, viral-classification diagnostics, and model interpretation components.


## Appendix A. Full Evidence Figure Index (Inside Same PDF)

This appendix contains additional supporting visuals so the report remains one self-contained grading artifact.


| Figure ID | Section | Interpretive Caption |
|---|---|---|
| `02_cleaning_fig_02_score_by_body_presence.png` | Cleaning | Box plots comparing score distribution for posts with vs. without body text, confirming that body missingness correlates with lower median engagement but higher variance among link/image posts. |
| `02_cleaning_fig_04_influence_plot.png` | Cleaning | Cook's Distance vs. leverage influence plot identifying high-influence observations in engagement regression diagnostics; points in the upper-right quadrant warrant individual inspection. |
| `02_cleaning_fig_05_qq_normality.png` | Cleaning | Q-Q plot assessing normality of score_log after log1p transformation; deviation in upper tail confirms residual heavy-tail behavior even post-transform. |
| `02_cleaning_fig_06_spearman_heatmap.png` | Cleaning | Rank-based correlation heatmap showing score_log and comms_num_log as the strongest correlated pair (rho = 0.79), motivating exclusion of both from predictor sets. |
| `03_eda_fig_02_comms_distribution.png` | EDA | Raw comment count distribution showing extreme right skew (median near 1, maximum > 10,000), paralleling the score distribution's heavy-tail pattern. |
| `03_eda_fig_04_comms_log_distribution.png` | EDA | Log-transformed comment count distribution showing approximate normality after log1p, validating the transformation choice for downstream parametric analysis. |
| `03_eda_fig_05_title_length_distribution.png` | EDA | Title character length distribution revealing a mode near 50-80 characters with a right tail of verbose posts; title length serves as a pre-publication text feature. |
| `03_eda_fig_06_hour_distribution.png` | EDA | Posting hour frequency showing peak activity during US afternoon/evening hours (14-22 UTC), consistent with retail trader schedules and after-market browsing. |
| `03_eda_fig_07_post_type_lumped_bar.png` | EDA | Frequency bar chart of lumped post types showing image and text as dominant categories, with link posts constituting a smaller but non-negligible fraction. |
| `03_eda_fig_08_post_type_bar.png` | EDA | Detailed post type breakdown including all original Reddit flair categories before lumping; validates the lumping strategy used in modeling. |
| `03_eda_fig_09_day_of_week_bar.png` | EDA | Day-of-week posting frequency showing relatively uniform weekday activity with slight weekend dips, suggesting limited day-of-week effects on engagement. |
| `03_eda_fig_11_title_score_scatter.png` | EDA | Scatter plot of title length vs. score_log showing weak association; title length alone is a poor predictor of engagement, motivating richer text features. |
| `03_eda_fig_12_score_hour_trend.png` | EDA | Mean score_log by posting hour with confidence bands; engagement peaks during late evening hours, supporting the late_night binary feature in the model. |
| `03_eda_fig_13_comments_hour_trend.png` | EDA | Mean comms_num_log by posting hour; comment engagement follows a similar hourly pattern to score, reinforcing the self-reinforcing engagement dynamic. |
| `03_eda_fig_14_score_dow_trend.png` | EDA | Mean score_log by day of week; minimal variation across days confirms that day-of-week was correctly excluded from the final feature set. |
| `03_eda_fig_15_daily_score_trend.png` | EDA | Daily mean score_log time series showing a dramatic spike during January 2021, visually confirming the temporal regime shift analyzed in Section 4.4. |
| `03_eda_fig_16_corr_heatmap.png` | EDA | Full correlation heatmap across all numeric variables; the strong score-comments cluster and weak associations with temporal features motivate feature group ordering. |
| `03_eda_fig_18_hour_day_heatmap.png` | EDA | Two-dimensional heatmap of mean engagement by hour and day of week; hotspots in weekday evenings confirm the interaction between temporal dimensions. |
| `04_feature_fig_02_topic_entropy_distribution.png` | Feature Engineering | Topic entropy distribution showing how concentrated vs. diffuse LDA topic assignments are; low-entropy posts have clear topical focus while high-entropy posts span multiple themes. |
| `04_feature_fig_03_dominant_topic_distribution.png` | Feature Engineering | Bar chart of dominant topic assignment frequency across the 5-topic LDA model; uneven distribution suggests some topics (likely GME-related) dominate the corpus. |
| `04_feature_fig_07_pred_prob_distribution.png` | Feature Engineering | Distribution of predicted viral probabilities showing strong class separation with most non-viral posts receiving low probabilities and viral posts spread across higher values. |
| `04_feature_fig_08_top_coefficients.png` | Feature Engineering | Top logistic regression coefficients identifying structural and temporal features as strongest predictors; sentiment and topic coefficients are comparatively small, supporting the ablation findings. |

![02_cleaning_fig_02_score_by_body_presence.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_02_score_by_body_presence.png)


![02_cleaning_fig_04_influence_plot.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_04_influence_plot.png)


![02_cleaning_fig_05_qq_normality.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_05_qq_normality.png)


![02_cleaning_fig_06_spearman_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_06_spearman_heatmap.png)


![03_eda_fig_02_comms_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_02_comms_distribution.png)


![03_eda_fig_04_comms_log_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_04_comms_log_distribution.png)


![03_eda_fig_05_title_length_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_05_title_length_distribution.png)


![03_eda_fig_06_hour_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_06_hour_distribution.png)


![03_eda_fig_07_post_type_lumped_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_07_post_type_lumped_bar.png)


![03_eda_fig_08_post_type_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_08_post_type_bar.png)


![03_eda_fig_09_day_of_week_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_09_day_of_week_bar.png)


![03_eda_fig_11_title_score_scatter.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_11_title_score_scatter.png)


![03_eda_fig_12_score_hour_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_12_score_hour_trend.png)


![03_eda_fig_13_comments_hour_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_13_comments_hour_trend.png)


![03_eda_fig_14_score_dow_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_14_score_dow_trend.png)


![03_eda_fig_15_daily_score_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_15_daily_score_trend.png)


![03_eda_fig_16_corr_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_16_corr_heatmap.png)


![03_eda_fig_18_hour_day_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_18_hour_day_heatmap.png)


![04_feature_fig_02_topic_entropy_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_02_topic_entropy_distribution.png)


![04_feature_fig_03_dominant_topic_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_03_dominant_topic_distribution.png)


![04_feature_fig_07_pred_prob_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_07_pred_prob_distribution.png)


![04_feature_fig_08_top_coefficients.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_08_top_coefficients.png)
